# Notebook 02 — Análise dos Indicadores
**Associação Passos Mágicos | PEDE 2022–2024**

Este notebook responde às 11 perguntas de negócio do Datathon, utilizando o dataset
unificado gerado no Notebook 01. Cada seção corresponde a uma pergunta específica,
com visualizações e interpretações orientadas a equipes gestoras.


## Configuração

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats

warnings.filterwarnings("ignore")
pd.set_option("display.float_format", "{:.3f}".format)

plt.rcParams.update({
    "figure.dpi": 120,
    "figure.facecolor": "white",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.family": "sans-serif",
})
PALETA = ["#1B4F72", "#2980B9", "#A9CCE3", "#F39C12", "#E74C3C", "#27AE60"]
sns.set_palette(PALETA)

BASE_DIR = os.path.dirname(os.getcwd())
DADOS_DIR = os.path.join(BASE_DIR, "data")
FIGURAS_DIR = os.path.join(BASE_DIR, "figuras")
os.makedirs(FIGURAS_DIR, exist_ok=True)

df = pd.read_csv(os.path.join(DADOS_DIR, "dados_limpos.csv"), encoding="utf-8-sig")
print(f"Dataset carregado: {df.shape[0]:,} registros | {df.shape[1]} variáveis")
print(f"Anos disponíveis: {sorted(df['ano'].unique())}")

ORDEM_PEDRA = ["Quartzo", "Ágata", "Ametista", "Topázio"]
COR_PEDRA = {
    "Quartzo":  "#5DADE2",
    "Ágata":    "#27AE60",
    "Ametista": "#8E44AD",
    "Topázio":  "#F39C12",
}


---
## Pergunta 1 — Perfil de Defasagem dos Alunos (IAN)
> *Qual é o perfil geral de defasagem dos alunos e como ele evolui ao longo dos anos?*

O **IAN (Indicador de Adequação ao Nível)** mede o quanto o aluno está alinhado ao nível
de ensino esperado para sua fase. Um valor baixo de IAN, combinado com defasagem negativa
(aluno abaixo do nível ideal), sinaliza necessidade de atenção reforçada.


In [ ]:
# Distribuição do nível de defasagem por ano
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Gráfico 1: Proporção de defasagem (negativa, zero, positiva)
def classif_defasagem(d):
    if pd.isna(d): return "Não informado"
    if d < 0: return "Defasado (abaixo do nível)"
    if d == 0: return "Adequado ao nível"
    return "Avançado (acima do nível)"

df["classif_defas"] = df["defasagem"].apply(classif_defasagem)
ordem_def = ["Defasado (abaixo do nível)", "Adequado ao nível",
             "Avançado (acima do nível)", "Não informado"]
cores_def = ["#E74C3C", "#27AE60", "#2980B9", "#BDC3C7"]

pivot_def = (
    df.groupby(["ano", "classif_defas"])
    .size()
    .reset_index(name="n")
    .assign(pct=lambda x: x.groupby("ano")["n"].transform(lambda s: s / s.sum() * 100))
    .pivot(index="classif_defas", columns="ano", values="pct")
    .reindex(ordem_def)
)
pivot_def.plot(kind="bar", ax=axes[0], color=["#1B4F72", "#2980B9", "#A9CCE3"],
               edgecolor="white", width=0.7)
axes[0].set_title("Classificação de Defasagem por Ano", fontweight="bold")
axes[0].set_ylabel("% dos alunos")
axes[0].tick_params(axis="x", rotation=20)
axes[0].legend(title="Ano")

# Gráfico 2: Média do IAN por ano e pedra
ian_pedra = df.groupby(["ano", "pedra_ref"])["ian"].mean().reset_index()
ian_pedra = ian_pedra[ian_pedra["pedra_ref"].isin(ORDEM_PEDRA)]
for pedra in ORDEM_PEDRA:
    subset = ian_pedra[ian_pedra["pedra_ref"] == pedra]
    axes[1].plot(subset["ano"], subset["ian"], marker="o",
                 label=pedra, color=COR_PEDRA.get(pedra, "gray"), linewidth=2)
axes[1].set_title("Média do IAN por Classificação e Ano", fontweight="bold")
axes[1].set_ylabel("IAN médio")
axes[1].set_xticks([2022, 2023, 2024])
axes[1].legend(title="Pedra")

plt.suptitle("Análise de Defasagem — IAN", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(FIGURAS_DIR, "05_ian_defasagem.png"), bbox_inches="tight")
plt.show()

# Resumo numérico
print("\nProporção de alunos defasados (abaixo do nível ideal) por ano:")
print(df[df["classif_defas"]=="Defasado (abaixo do nível)"].groupby("ano").size()
      .div(df.groupby("ano").size()) * 100)


---
## Pergunta 2 — Desempenho Acadêmico (IDA)
> *O desempenho acadêmico médio (IDA) está melhorando, estagnado ou caindo ao longo
das fases e anos?*

O **IDA (Indicador de Aprendizagem)** agrega as notas de Português, Matemática e Inglês.
Avaliar sua trajetória por fase e por ano revela se o programa está gerando ganhos
de aprendizagem sustentáveis.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Evolução do IDA médio por ano
ida_ano = df.groupby("ano")["ida"].agg(["mean", "median", "std"]).reset_index()
axes[0].fill_between(ida_ano["ano"],
                     ida_ano["mean"] - ida_ano["std"],
                     ida_ano["mean"] + ida_ano["std"],
                     alpha=0.2, color="#2980B9")
axes[0].plot(ida_ano["ano"], ida_ano["mean"], marker="o", color="#1B4F72",
             linewidth=2.5, label="Média")
axes[0].plot(ida_ano["ano"], ida_ano["median"], marker="s", color="#E74C3C",
             linewidth=2, linestyle="--", label="Mediana")
for _, row in ida_ano.iterrows():
    axes[0].annotate(f"{row['mean']:.2f}", (row["ano"], row["mean"]),
                     textcoords="offset points", xytext=(0, 10), ha="center", fontsize=9)
axes[0].set_title("Evolução do IDA Médio (2022–2024)", fontweight="bold")
axes[0].set_ylabel("IDA")
axes[0].set_xticks([2022, 2023, 2024])
axes[0].legend()

# IDA por Pedra e ano (box plot)
df_pedra_ord = df[df["pedra_ref"].isin(ORDEM_PEDRA)].copy()
df_pedra_ord["pedra_ref"] = pd.Categorical(df_pedra_ord["pedra_ref"],
                                            categories=ORDEM_PEDRA, ordered=True)
sns.boxplot(data=df_pedra_ord, x="pedra_ref", y="ida", hue="ano",
            palette=["#1B4F72", "#2980B9", "#A9CCE3"], ax=axes[1],
            order=ORDEM_PEDRA, linewidth=0.8)
axes[1].set_title("Distribuição do IDA por Classificação e Ano", fontweight="bold")
axes[1].set_ylabel("IDA")
axes[1].set_xlabel("Classificação (Pedra)")
axes[1].legend(title="Ano", fontsize=8)

plt.suptitle("Análise do Indicador de Aprendizagem — IDA", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(FIGURAS_DIR, "06_ida_desempenho.png"), bbox_inches="tight")
plt.show()

print("\nMédia do IDA por ano:")
print(ida_ano[["ano", "mean", "median"]].rename(
    columns={"mean": "Média", "median": "Mediana"}).to_string(index=False))


---
## Pergunta 3 — Engajamento (IEG) × Desempenho e Ponto de Virada
> *O grau de engajamento (IEG) tem relação direta com IDA e IPV?*

Alunos mais engajados tendem a apresentar melhor desempenho? E qual o papel do
engajamento no alcance do Ponto de Virada (IPV)?


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# IEG × IDA (scatter com tendência)
sample = df[["ieg", "ida", "ipv", "ano"]].dropna()
axes[0].scatter(sample["ieg"], sample["ida"], alpha=0.25, s=15, color="#2980B9")
z = np.polyfit(sample["ieg"], sample["ida"], 1)
x_line = np.linspace(sample["ieg"].min(), sample["ieg"].max(), 100)
axes[0].plot(x_line, np.poly1d(z)(x_line), color="#E74C3C", linewidth=2, label="Tendência")
r, p = stats.pearsonr(sample["ieg"].dropna(), sample["ida"].dropna())
axes[0].set_title(f"IEG × IDA  (r = {r:.2f}, p {'< 0.001' if p < 0.001 else f'= {p:.3f}'})",
                  fontweight="bold")
axes[0].set_xlabel("IEG (Engajamento)")
axes[0].set_ylabel("IDA (Aprendizagem)")
axes[0].legend()

# IEG × IPV
axes[1].scatter(sample["ieg"], sample["ipv"], alpha=0.25, s=15, color="#27AE60")
z2 = np.polyfit(sample["ieg"].dropna(), sample["ipv"].dropna(), 1)
axes[1].plot(x_line, np.poly1d(z2)(x_line), color="#E74C3C", linewidth=2,
             label="Tendência")
r2, p2 = stats.pearsonr(sample["ieg"].dropna(), sample["ipv"].dropna())
axes[1].set_title(f"IEG × IPV  (r = {r2:.2f}, p {'< 0.001' if p2 < 0.001 else f'= {p2:.3f}'})",
                  fontweight="bold")
axes[1].set_xlabel("IEG (Engajamento)")
axes[1].set_ylabel("IPV (Ponto de Virada)")
axes[1].legend()

plt.suptitle("Relação entre Engajamento (IEG) e Desempenho", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(FIGURAS_DIR, "07_ieg_vs_ida_ipv.png"), bbox_inches="tight")
plt.show()


---
## Pergunta 4 — Coerência da Autoavaliação (IAA)
> *As percepções dos alunos sobre si mesmos (IAA) são coerentes com seu desempenho real
(IDA) e engajamento (IEG)?*

Uma divergência consistente entre autoavaliação e indicadores objetivos pode indicar
baixa autopercepção ou excesso de confiança, ambas situações que exigem intervenção.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sample = df[["iaa", "ida", "ieg", "pedra_ref"]].dropna()
sample = sample[sample["pedra_ref"].isin(ORDEM_PEDRA)]

# IAA × IDA por classificação
for pedra in ORDEM_PEDRA:
    sub = sample[sample["pedra_ref"] == pedra]
    axes[0].scatter(sub["iaa"], sub["ida"], alpha=0.4, s=15,
                    label=pedra, color=COR_PEDRA[pedra])
axes[0].plot([0, 10], [0, 10], "k--", linewidth=1, alpha=0.5, label="Linha ideal (IAA=IDA)")
axes[0].set_xlabel("IAA (Autoavaliação)")
axes[0].set_ylabel("IDA (Aprendizagem)")
axes[0].set_title("Autoavaliação × Desempenho Real", fontweight="bold")
axes[0].legend(fontsize=8, title="Pedra")

# Diferença IAA - IDA por ano (média)
df["diff_iaa_ida"] = df["iaa"] - df["ida"]
diff_ano = df.groupby(["ano", "pedra_ref"])["diff_iaa_ida"].mean().reset_index()
diff_ano = diff_ano[diff_ano["pedra_ref"].isin(ORDEM_PEDRA)]

diff_pivot = diff_ano.pivot(index="pedra_ref", columns="ano",
                             values="diff_iaa_ida").reindex(ORDEM_PEDRA)
diff_pivot.plot(kind="bar", ax=axes[1], color=["#1B4F72", "#2980B9", "#A9CCE3"],
                edgecolor="white", width=0.7)
axes[1].axhline(0, color="black", linewidth=1, linestyle="--")
axes[1].set_title("Diferença Média (IAA − IDA) por Pedra", fontweight="bold")
axes[1].set_ylabel("Diferença (IAA − IDA)")
axes[1].set_xlabel("Classificação")
axes[1].tick_params(axis="x", rotation=0)
axes[1].legend(title="Ano", fontsize=8)

plt.suptitle("Coerência da Autoavaliação", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(FIGURAS_DIR, "08_iaa_coerencia.png"), bbox_inches="tight")
plt.show()

print("\nDiferença média IAA − IDA por ano (positivo = superestimação):")
print(df.groupby("ano")["diff_iaa_ida"].agg(["mean", "median"]).round(3))


---
## Pergunta 5 — Padrões Psicossociais (IPS)
> *Há padrões psicossociais que antecedem quedas de desempenho ou engajamento?*

O **IPS** captura dimensões socioemocionais e de bem-estar. Identificar correlações entre
IPS baixo e quedas futuras de IDA ou IEG permite antecipar intervenções da equipe de psicologia.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# IPS × IDA
sample_ps = df[["ips", "ida", "ieg", "pedra_ref"]].dropna()
sample_ps = sample_ps[sample_ps["pedra_ref"].isin(ORDEM_PEDRA)]
for pedra in ORDEM_PEDRA:
    sub = sample_ps[sample_ps["pedra_ref"] == pedra]
    axes[0].scatter(sub["ips"], sub["ida"], alpha=0.35, s=15,
                    label=pedra, color=COR_PEDRA[pedra])
z_ips = np.polyfit(sample_ps["ips"], sample_ps["ida"], 1)
x_ips = np.linspace(sample_ps["ips"].min(), sample_ps["ips"].max(), 100)
axes[0].plot(x_ips, np.poly1d(z_ips)(x_ips), "k-", linewidth=2, label="Tendência")
r_ips, _ = stats.pearsonr(sample_ps["ips"], sample_ps["ida"])
axes[0].set_title(f"IPS × IDA  (r = {r_ips:.2f})", fontweight="bold")
axes[0].set_xlabel("IPS (Psicossocial)")
axes[0].set_ylabel("IDA (Aprendizagem)")
axes[0].legend(fontsize=8, title="Pedra")

# Distribuição IPS por classificação (violin)
df_ord = df[df["pedra_ref"].isin(ORDEM_PEDRA)].copy()
df_ord["pedra_ref"] = pd.Categorical(df_ord["pedra_ref"],
                                      categories=ORDEM_PEDRA, ordered=True)
sns.violinplot(data=df_ord, x="pedra_ref", y="ips", hue="pedra_ref",
               palette=list(COR_PEDRA.values()), ax=axes[1],
               order=ORDEM_PEDRA, legend=False, inner="quartile")
axes[1].set_title("Distribuição do IPS por Classificação", fontweight="bold")
axes[1].set_xlabel("Classificação (Pedra)")
axes[1].set_ylabel("IPS (Psicossocial)")

plt.suptitle("Padrões Psicossociais e Desempenho", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(FIGURAS_DIR, "09_ips_padroes.png"), bbox_inches="tight")
plt.show()


---
## Pergunta 6 — Avaliação Psicopedagógica (IPP) × Defasagem (IAN)
> *As avaliações psicopedagógicas (IPP) confirmam ou contradizem a defasagem identificada pelo IAN?*

O IPP e o IAN medem dimensões complementares: o IPP é uma avaliação qualitativa da equipe
pedagógica, enquanto o IAN é calculado a partir do nível esperado versus alcançado.


In [ ]:
df_ipp = df[["ipp", "ian", "defasagem", "pedra_ref"]].dropna(subset=["ipp", "ian"])
df_ipp = df_ipp[df_ipp["pedra_ref"].isin(ORDEM_PEDRA)]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# IPP × IAN scatter
for pedra in ORDEM_PEDRA:
    sub = df_ipp[df_ipp["pedra_ref"] == pedra]
    axes[0].scatter(sub["ipp"], sub["ian"], alpha=0.4, s=20,
                    label=pedra, color=COR_PEDRA[pedra])
axes[0].plot([0, 10], [0, 10], "k--", linewidth=1, alpha=0.5)
r_ipp_ian, _ = stats.pearsonr(df_ipp["ipp"], df_ipp["ian"])
axes[0].set_title(f"IPP × IAN  (r = {r_ipp_ian:.2f})", fontweight="bold")
axes[0].set_xlabel("IPP (Psicopedagógico)")
axes[0].set_ylabel("IAN (Adequação ao Nível)")
axes[0].legend(fontsize=8, title="Pedra")

# IPP por grupo de defasagem
df_ipp["grupo_defas"] = pd.cut(df_ipp["defasagem"].fillna(0),
                                bins=[-99, -1, 0, 99],
                                labels=["Defasado", "Adequado", "Avançado"])
ipp_group = df_ipp.groupby("grupo_defas")["ipp"].mean()
bars = axes[1].bar(ipp_group.index, ipp_group.values,
                   color=["#E74C3C", "#27AE60", "#2980B9"], edgecolor="white", width=0.5)
axes[1].set_title("IPP Médio por Grupo de Defasagem", fontweight="bold")
axes[1].set_ylabel("IPP Médio")
axes[1].set_xlabel("Grupo de Defasagem")
for bar, v in zip(bars, ipp_group.values):
    axes[1].text(bar.get_x() + bar.get_width() / 2, v + 0.1,
                 f"{v:.2f}", ha="center", va="bottom", fontweight="bold")

plt.suptitle("Convergência entre IPP e IAN", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(FIGURAS_DIR, "10_ipp_ian.png"), bbox_inches="tight")
plt.show()


---
## Pergunta 7 — Fatores que Influenciam o Ponto de Virada (IPV)
> *Quais comportamentos — acadêmicos, emocionais ou de engajamento — mais influenciam o IPV?*

O **IPV (Ponto de Virada)** representa uma mudança qualitativa na trajetória do aluno.
Entender quais indicadores mais o predizem orienta as estratégias de mentoria.


In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler

FEATURES_IPV = ["iaa", "ieg", "ips", "ida", "ian", "ipp"]
df_ipv = df[FEATURES_IPV + ["ipv"]].dropna()

X = df_ipv[FEATURES_IPV]
y = df_ipv["ipv"]

rf = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
rf.fit(X, y)

importancias = pd.Series(rf.feature_importances_, index=FEATURES_IPV).sort_values()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Importância de variáveis
importancias.plot.barh(ax=axes[0], color="#2980B9", edgecolor="white")
axes[0].set_title("Importância dos Indicadores para o IPV (Random Forest)", fontweight="bold")
axes[0].set_xlabel("Importância relativa")
for i, v in enumerate(importancias):
    axes[0].text(v + 0.002, i, f"{v:.3f}", va="center", fontsize=9)

# Correlações de Pearson com IPV
corrs = df_ipv[FEATURES_IPV].corrwith(df_ipv["ipv"]).sort_values()
colors_corr = ["#E74C3C" if v < 0 else "#27AE60" for v in corrs]
corrs.plot.barh(ax=axes[1], color=colors_corr, edgecolor="white")
axes[1].axvline(0, color="black", linewidth=0.8)
axes[1].set_title("Correlação de Pearson com o IPV", fontweight="bold")
axes[1].set_xlabel("Correlação (r)")
for i, v in enumerate(corrs):
    axes[1].text(v + (0.005 if v >= 0 else -0.025), i, f"{v:.2f}",
                 va="center", fontsize=9)

plt.suptitle("Fatores que Influenciam o Ponto de Virada (IPV)",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(FIGURAS_DIR, "11_ipv_fatores.png"), bbox_inches="tight")
plt.show()

print(f"\nR² do modelo Random Forest no IPV: {rf.score(X, y):.3f}")


---
## Pergunta 8 — Multidimensionalidade: Combinações que Elevam o INDE
> *Quais combinações de indicadores (IDA + IEG + IPS + IPP) elevam mais o INDE?*

O INDE é calculado pela média ponderada de todos os indicadores. Identificar quais
combinações resultam nas maiores notas globais orienta o foco das intervenções.


In [ ]:
FEATURES_INDE = ["iaa", "ieg", "ips", "ida", "ipv", "ian", "ipp"]
df_inde = df[FEATURES_INDE + ["inde"]].dropna()

# Correlação com INDE
corrs_inde = df_inde[FEATURES_INDE].corrwith(df_inde["inde"]).sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Barras de correlação
colors = [COR_PEDRA["Topázio"] if v > 0.5 else
          COR_PEDRA["Ametista"] if v > 0.35 else
          COR_PEDRA["Ágata"] for v in corrs_inde]
axes[0].barh(corrs_inde.index[::-1], corrs_inde.values[::-1],
             color=colors[::-1], edgecolor="white")
axes[0].set_title("Correlação dos Indicadores com o INDE", fontweight="bold")
axes[0].set_xlabel("Correlação de Pearson (r)")
for i, v in enumerate(corrs_inde.values[::-1]):
    axes[0].text(v + 0.005, i, f"{v:.2f}", va="center", fontsize=9)

# Heatmap por quartil de INDE
df_inde["quartil_inde"] = pd.qcut(df_inde["inde"], 4,
                                   labels=["Q1 (mais baixo)", "Q2", "Q3", "Q4 (mais alto)"])
medias_quartil = df_inde.groupby("quartil_inde")[FEATURES_INDE].mean()
sns.heatmap(medias_quartil.T, annot=True, fmt=".1f", cmap="Blues",
            linewidths=0.5, ax=axes[1], annot_kws={"size": 9})
axes[1].set_title("Média dos Indicadores por Quartil de INDE", fontweight="bold")
axes[1].set_xlabel("Quartil do INDE")

plt.suptitle("Multidimensionalidade: O que Eleva o INDE?", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(FIGURAS_DIR, "12_inde_multidimensional.png"), bbox_inches="tight")
plt.show()

print("\nCorrelação com INDE:")
for ind, r in corrs_inde.items():
    print(f"  {ind.upper()}: r = {r:.3f}")


---
## Pergunta 9 — Previsão de Risco com Machine Learning
> Ver **Notebook 03 — Modelo Preditivo**, que trata exclusivamente desta questão.


---
## Pergunta 10 — Efetividade do Programa por Fase (Pedra)
> *Os indicadores mostram melhora consistente ao longo do ciclo nas diferentes fases
(Quartzo → Topázio), confirmando o impacto real do programa?*


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

INDICADORES_EFF = ["inde", "ida", "ieg", "ipv"]
NOMES_EFF = {
    "inde": "INDE (Global)", "ida": "IDA (Aprendizagem)",
    "ieg": "IEG (Engajamento)", "ipv": "IPV (Ponto de Virada)"
}

df_eff = df[df["pedra_ref"].isin(ORDEM_PEDRA)].copy()
df_eff["pedra_ref"] = pd.Categorical(df_eff["pedra_ref"],
                                      categories=ORDEM_PEDRA, ordered=True)

for ax, ind in zip(axes.flatten(), INDICADORES_EFF):
    medias = df_eff.groupby(["pedra_ref", "ano"])[ind].mean().reset_index()
    for ano in sorted(df_eff["ano"].unique()):
        sub = medias[medias["ano"] == ano]
        ax.plot(sub["pedra_ref"].astype(str), sub[ind], marker="o",
                label=str(ano), linewidth=2)
    ax.set_title(NOMES_EFF[ind], fontweight="bold")
    ax.set_ylabel("Média")
    ax.set_xlabel("Classificação (Pedra)")
    ax.legend(title="Ano", fontsize=8)

plt.suptitle("Efetividade do Programa: Progressão por Classificação",
             fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(FIGURAS_DIR, "13_efetividade_programa.png"), bbox_inches="tight")
plt.show()

# Tabela de síntese
print("\nMédia dos indicadores por Pedra e Ano:")
tabela_eff = df_eff.groupby(["pedra_ref", "ano"])[INDICADORES_EFF].mean().round(2)
print(tabela_eff.to_string())


---
## Pergunta 11 — Insights Adicionais e Oportunidades de Melhoria


In [ ]:
# ─── Insight 1: Gênero e desempenho ───────────────────────────
df_gen = df[df["genero"].isin(["Feminino", "Masculino"])].copy()
genero_media = df_gen.groupby(["genero", "ano"])[["inde", "ida", "ieg"]].mean()
print("=== Insight 1: Indicadores médios por gênero e ano ===")
print(genero_media.round(3))


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, ind, titulo in zip(axes, ["inde", "ida", "ieg"],
                            ["INDE", "IDA", "IEG"]):
    data = df_gen.groupby(["ano", "genero"])[ind].mean().reset_index()
    for genero, grupo in data.groupby("genero"):
        cor = "#E74C3C" if genero == "Feminino" else "#2980B9"
        ax.plot(grupo["ano"], grupo[ind], marker="o", label=genero,
                color=cor, linewidth=2)
    ax.set_title(f"{titulo} médio por Gênero", fontweight="bold")
    ax.set_ylabel(titulo)
    ax.set_xticks([2022, 2023, 2024])
    ax.legend(title="Gênero", fontsize=8)

plt.suptitle("Insight 1: Desempenho por Gênero (2022–2024)",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(FIGURAS_DIR, "14_insight_genero.png"), bbox_inches="tight")
plt.show()


In [ ]:
# ─── Insight 2: Escola pública vs privada ──────────────────
df_esc = df[df["instituicao"].isin(["Escola Pública", "Escola Privada",
                                     "Pública", "Privada",
                                     "Escola P\u00fablica", "P\u00fablica"])].copy()
df_esc["tipo_escola"] = df_esc["instituicao"].apply(
    lambda x: "Pública" if "blica" in str(x) else "Privada"
)
esc_media = df_esc.groupby(["tipo_escola", "ano"])[["inde", "ida"]].mean()
print("\n=== Insight 2: Desempenho por tipo de escola ===")
print(esc_media.round(3))


In [ ]:
# ─── Insight 3: Alunos veteranos vs ingressantes ──────────
df["veterano"] = (df["anos_na_pm"] >= 2).map({True: "Veterano (≥2 anos)",
                                               False: "Ingressante (<2 anos)"})
vet_media = df.groupby(["veterano", "ano"])[["inde", "ieg", "ipv"]].mean()
print("\n=== Insight 3: Veteranos vs Ingressantes ===")
print(vet_media.round(3))

fig, ax = plt.subplots(figsize=(10, 5))
for grupo, dados in df.groupby("veterano"):
    media = dados.groupby("ano")["inde"].mean()
    ax.plot(media.index, media.values, marker="o",
            label=grupo, linewidth=2)
ax.set_title("INDE Médio: Veteranos vs Ingressantes", fontweight="bold")
ax.set_ylabel("INDE médio")
ax.set_xticks([2022, 2023, 2024])
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(FIGURAS_DIR, "15_insight_veteranos.png"), bbox_inches="tight")
plt.show()


In [ ]:
# ─── Insight 4: Análise dos indicados para bolsa ──────────
df_bolsa = df[df["indicado_bolsa"].notna()].copy()
df_bolsa["indicado_bolsa"] = df_bolsa["indicado_bolsa"].astype(int)
bolsa_ind = df_bolsa.groupby("indicado_bolsa")[
    ["inde", "ida", "ieg", "iaa", "ipv"]
].mean()
bolsa_ind.index = ["Não indicado", "Indicado para bolsa"]
print("\n=== Insight 4: Perfil dos indicados para bolsa ===")
print(bolsa_ind.round(3))
